In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('new_data.csv')


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119135 entries, 0 to 119134
Data columns (total 23 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   video_id                119135 non-null  object 
 1   date                    119135 non-null  object 
 2   views                   119135 non-null  int64  
 3   likes                   119135 non-null  int64  
 4   comments                119135 non-null  int64  
 5   watch_time_minutes      119135 non-null  float64
 6   video_length_minutes    119135 non-null  float64
 7   subscribers             119135 non-null  int64  
 8   ad_revenue_usd          119135 non-null  float64
 9   engagement_rate         119135 non-null  float64
 10  category_Entertainment  119135 non-null  bool   
 11  category_Gaming         119135 non-null  bool   
 12  category_Lifestyle      119135 non-null  bool   
 13  category_Music          119135 non-null  bool   
 14  category_Tech       

In [5]:
# Select the features we want to use for training
# target: 'ad_revenue_usd'
features = ['views', 'likes', 'comments', 'watch_time_minutes', 'video_length_minutes', 'subscribers', 'category_Entertainment','category_Gaming','category_Lifestyle','category_Music','category_Tech','country_CA','country_DE','country_IN','country_UK','country_US'              
]
target = 'ad_revenue_usd'

# Create final dataframe
model_df = df[features + [target]].copy()
model_df.to_csv('new_data.csv', index=False)
print(f"Cleaned dataset saved with {len(model_df)} records.")


Cleaned dataset saved with 119135 records.


In [6]:
X = model_df[features]
y = model_df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ['views', 'likes', 'comments', 'watch_time_minutes', 'video_length_minutes', 'subscribers']
categorical_features = ['category_Entertainment','category_Gaming','category_Lifestyle','category_Music','category_Tech','country_CA','country_DE','country_IN','country_UK','country_US'              
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', 'passthrough', categorical_features) # using codes as provided
    ])


In [7]:
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(),
    'Lasso Regression': Lasso(),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=50, random_state=42)
}

results = []
trained_pipelines = {}

for name, model in models.items():
    print(f"Training {name}...")
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                               ('regressor', model)])
    
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    results.append({'Model': name, 'R2 Score': r2, 'RMSE': rmse, 'MAE': mae})
    trained_pipelines[name] = pipeline

results_df = pd.DataFrame(results).sort_values(by='R2 Score', ascending=False)
results_df

Training Linear Regression...
Training Ridge Regression...
Training Lasso Regression...
Training Random Forest...
Training Gradient Boosting...


,Model,R2 Score,RMSE,MAE
1,Ridge Regression,0.949514,13.928588,3.249071
0,Linear Regression,0.949514,13.928589,3.249091
2,Lasso Regression,0.948743,14.034524,4.477799
4,Gradient Boosting,0.948485,14.069783,4.536029
3,Random Forest,0.945716,14.442993,3.929360


In [8]:
best_model_name = results_df.iloc[0]['Model']
print(f"Best Model: {best_model_name}")
best_pipeline = trained_pipelines[best_model_name]

joblib.dump(best_pipeline, 'model.pkl')
print("Best model saved as 'model.pkl'.")

Best Model: Ridge Regression
Best model saved as 'model.pkl'.
